# 00 — Source Sold eBay Listings (Finding API)

Collects sold/completed eBay GB listings across target categories using the
eBay Finding API (`findCompletedItems` + `SoldItemsOnly=true`).
The sale price is the training label for the XGBoost pricing model.

**Output**
- `datasets/ebay_sold_listings.csv`
- `datasets/ebay_sold_listings.parquet`
- `datasets/ebay_{slug}.csv` — one file per category

**Requirements** — add to `.env`:
```
EBAY_CLIENT_ID=your-production-app-id
EBAY_ENV=production
```


In [9]:
%pip install pandas pyarrow python-dotenv httpx --quiet


[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import base64
import os
import time
import urllib.parse
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv

load_dotenv(Path("../.env"))


In [ ]:
EBAY_APP_ID = os.getenv("EBAY_CLIENT_ID", "")   # Finding API uses App ID directly, no OAuth
EBAY_ENV    = os.getenv("EBAY_ENV", "production")
GLOBAL_ID   = "EBAY-GB"

_FINDING_URLS = {
    "sandbox":    "https://svcs.sandbox.ebay.com/services/search/FindingService/v1",
    "production": "https://svcs.ebay.com/services/search/FindingService/v1",
}

FINDING_URL  = _FINDING_URLS[EBAY_ENV]
DATASETS_DIR = Path("../datasets")
DATASETS_DIR.mkdir(exist_ok=True)

assert EBAY_APP_ID, "EBAY_CLIENT_ID missing from .env"

print(f"eBay env    : {EBAY_ENV}")
print(f"Global ID   : {GLOBAL_ID}")
print(f"Finding URL : {FINDING_URL}")
print(f"Datasets dir: {DATASETS_DIR.resolve()}")


## Finding API helpers

`findCompletedItems` with `SoldItemsOnly=true` returns listings that ended with a sale.
The `currentPrice` field is the actual sale price — the training label.


In [ ]:
def _finding_get(params: dict) -> httpx.Response:
    # httpx percent-encodes '(' and ')' in param names which breaks the Finding API.
    # Build the query string manually — encode values only, leave key chars literal.
    qs = "&".join(f"{k}={urllib.parse.quote(str(v), safe='')}" for k, v in params.items())
    r = httpx.get(f"{FINDING_URL}?{qs}", timeout=30)
    if not r.is_success:
        print(f"  eBay error response: {r.text[:500]}")
    r.raise_for_status()
    return r


def search_sold_page(query: str, page: int = 1, entries: int = 100) -> tuple[list[dict], int]:
    r = _finding_get({
        "OPERATION-NAME":                "findCompletedItems",
        "SERVICE-VERSION":               "1.0.0",
        "SECURITY-APPNAME":              EBAY_APP_ID,
        "RESPONSE-DATA-FORMAT":          "JSON",
        "GLOBAL-ID":                     GLOBAL_ID,
        "keywords":                      query.strip(),
        "itemFilter(0).name":            "SoldItemsOnly",
        "itemFilter(0).value":           "true",
        "outputSelector(0)":             "SellerInfo",
        "sortOrder":                     "EndTimeSoonest",
        "paginationInput.entriesPerPage": min(entries, 100),
        "paginationInput.pageNumber":    page,
    })
    resp  = r.json().get("findCompletedItemsResponse", [{}])[0]
    items = resp.get("searchResult", [{}])[0].get("item", [])
    pages = int(resp.get("paginationOutput", [{}])[0].get("totalPages", ["1"])[0])
    return items, pages


def _first(d: dict, key: str, default=""):
    val = d.get(key, [default])
    return val[0] if val else default


def flatten_sold_item(item: dict, category_slug: str) -> dict | None:
    selling = _first(item, "sellingStatus", {})
    if not isinstance(selling, dict):
        return None
    if _first(selling, "sellingState") != "EndedWithSales":
        return None

    price_obj   = _first(selling, "currentPrice", {})
    price_value = float(price_obj.get("__value__", 0)) if isinstance(price_obj, dict) else 0.0
    currency    = price_obj.get("@currencyId", "") if isinstance(price_obj, dict) else ""

    condition    = _first(item, "condition", {})
    listing_info = _first(item, "listingInfo", {})
    primary_cat  = _first(item, "primaryCategory", {})
    seller       = _first(item, "seller", {})

    try:
        feedback_score = int(_first(seller, "feedbackScore")) if isinstance(seller, dict) else None
    except (ValueError, TypeError):
        feedback_score = None
    try:
        feedback_pct = float(_first(seller, "positiveFeedbackPercent")) if isinstance(seller, dict) else None
    except (ValueError, TypeError):
        feedback_pct = None

    return {
        "item_id":               _first(item, "itemId"),
        "title":                 _first(item, "title"),
        "category":              category_slug,
        "ebay_category":         _first(primary_cat, "categoryName") if isinstance(primary_cat, dict) else "",
        "price":                 price_value,
        "currency":              currency,
        "condition":             _first(condition, "conditionDisplayName") if isinstance(condition, dict) else "",
        "condition_id":          _first(condition, "conditionId") if isinstance(condition, dict) else "",
        "listing_type":          _first(listing_info, "listingType") if isinstance(listing_info, dict) else "",
        "seller_feedback_score": feedback_score,
        "seller_feedback_pct":   feedback_pct,
        "location_country":      _first(item, "country"),
        "location":              _first(item, "location"),
        "image_url":             _first(item, "galleryURL"),
        "listing_url":           _first(item, "viewItemURL"),
        "sold_at":               _first(listing_info, "endTime") if isinstance(listing_info, dict) else "",
    }


# Sanity check
_test_items, _test_pages = search_sold_page("iphone", page=1, entries=3)
print(f"Sanity check: got {len(_test_items)} items, {_test_pages} total pages")
if _test_items:
    print("First item title:", _first(_test_items[0], "title"))


## Categories

## Categories

Five high-volume second-hand categories on eBay GB.

In [20]:
CATEGORIES = [
    # Smartphones & wearables — strong brand premium, fast age depreciation
    {"slug": "smartphones",       "query": "iphone samsung galaxy mobile phone"},
    {"slug": "smartwatches",      "query": "apple watch smartwatch fitness tracker"},

    # Computing — brand + age critical, $100-$2000
    {"slug": "laptops",           "query": "laptop macbook notebook computer"},
    {"slug": "tablets",           "query": "ipad tablet android samsung tab"},

    # Audio — budget to premium brand spread
    {"slug": "headphones",        "query": "wireless headphones earbuds sony bose"},
    {"slug": "speakers",          "query": "bluetooth speaker portable"},

    # Gaming — condition + completeness drive large price gaps
    {"slug": "consoles",          "query": "playstation xbox nintendo switch console"},
    {"slug": "video_games",       "query": "PS5 Xbox Nintendo switch game"},

    # Fashion — brand encoding across wide range, condition less harsh than electronics
    {"slug": "trainers",          "query": "nike adidas trainers sneakers shoes"},
    {"slug": "mens_clothing",     "query": "mens jacket hoodie jeans shirt"},
    {"slug": "womens_clothing",   "query": "womens dress jacket blouse top"},
    {"slug": "handbags",          "query": "handbag designer coach gucci louis vuitton"},
    {"slug": "vintage_clothing",  "query": "vintage levi ralph lauren retro clothing"},

    # Home & kitchen — weak vs. strong brand contrast (air fryer vs. Dyson/KitchenAid)
    {"slug": "vacuums",           "query": "dyson hoover vacuum cleaner"},
    {"slug": "kitchen_appliances","query": "air fryer microwave blender kitchenaid"},
    {"slug": "coffee",            "query": "coffee machine espresso nespresso"},

    # Sports & outdoors — condition + brand, $15-$600
    {"slug": "golf",              "query": "golf clubs driver irons putter"},
    {"slug": "cycling",           "query": "road bike mountain bike cycling"},
    {"slug": "gym_equipment",     "query": "dumbbells weights kettlebell gym"},

    # Collectibles — condition + rarity dominate, huge price spread $5-$2000+
    {"slug": "pokemon_cards",     "query": "pokemon cards trading card game"},
    {"slug": "lego",              "query": "lego set creator technic star wars"},
    {"slug": "vinyl_records",     "query": "vinyl record LP album"},
    {"slug": "cameras",           "query": "film camera vintage canon nikon"},
    {"slug": "funko_pop",         "query": "funko pop figure vinyl"},

    # Books & media — low price anchor, low condition sensitivity
    {"slug": "textbooks",         "query": "university textbook academic study"},
    {"slug": "books",             "query": "hardcover novel fiction non-fiction book"},
]

ITEMS_PER_CATEGORY  = 500
PAGE_SIZE           = 200
SLEEP_BETWEEN_PAGES = 0.4

print(f"Target: {ITEMS_PER_CATEGORY} x {len(CATEGORIES)} = {ITEMS_PER_CATEGORY * len(CATEGORIES):,} rows")


Target: 500 x 26 = 13,000 rows


## Scrape

In [ ]:
all_rows: list[dict] = []

for cat in CATEGORIES:
    slug  = cat["slug"]
    query = cat["query"]
    rows: list[dict] = []
    page  = 1

    print(f"\n{chr(9472) * 55}")
    print(f"Category : {slug}")
    print(f"Query    : {query!r}")

    while len(rows) < ITEMS_PER_CATEGORY:
        try:
            items, total_pages = search_sold_page(query, page=page)
        except httpx.HTTPStatusError as exc:
            body = exc.response.text
            if "10001" in body or "RateLimiter" in body:
                print(f"  Rate limit hit on page={page} — stopping entire scrape.")
                print(f"  Wait for quota reset (midnight UTC) then re-run.")
                all_rows.extend(rows[:ITEMS_PER_CATEGORY])
                raise SystemExit("Rate limited") from exc
            print(f"  HTTP {exc.response.status_code} on page={page}: {body[:200]} -- skipping category")
            break
        except httpx.ConnectError as exc:
            print(f"  Connection error on page={page}: {exc} -- skipping category")
            break

        if not items:
            print(f"  No more results at page={page}")
            break

        for item in items:
            flat = flatten_sold_item(item, slug)
            if flat is not None:
                rows.append(flat)

        print(f"  page={page:>3}/{total_pages}  page_items={len(items):>3}  collected={len(rows):>4}")

        if page >= total_pages or page >= 100:
            print(f"  Reached last page ({page})")
            break

        page += 1
        time.sleep(SLEEP_BETWEEN_PAGES)

    kept = rows[:ITEMS_PER_CATEGORY]
    all_rows.extend(kept)
    print(f"  kept {len(kept)} sold items")

print(f"\nTotal rows (before dedup): {len(all_rows):,}")


## Build DataFrame & save

In [ ]:
df = pd.DataFrame(all_rows)
df = df.drop_duplicates(subset=["item_id"])
df["price"]                = pd.to_numeric(df["price"],                errors="coerce")
df["seller_feedback_score"] = pd.to_numeric(df["seller_feedback_score"], errors="coerce")
df["seller_feedback_pct"]  = pd.to_numeric(df["seller_feedback_pct"],  errors="coerce")
df["sold_at"]              = pd.to_datetime(df["sold_at"],             utc=True, errors="coerce")

print(f"Shape after dedup : {df.shape}")
print(f"Sold date range   : {df['sold_at'].min()} → {df['sold_at'].max()}")
df.head(3)


In [ ]:
csv_path = DATASETS_DIR / "ebay_sold_listings.csv"
df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}  ({csv_path.stat().st_size / 1024:.0f} KB)")

try:
    parquet_path = DATASETS_DIR / "ebay_sold_listings.parquet"
    df.to_parquet(parquet_path, index=False)
    print(f"Saved: {parquet_path}  ({parquet_path.stat().st_size / 1024:.0f} KB)")
except Exception as exc:
    print(f"Parquet skipped ({exc})")

for slug, group in df.groupby("category"):
    path = DATASETS_DIR / f"ebay_{slug}.csv"
    group.to_csv(path, index=False)
    print(f"Saved: {path}  ({len(group)} rows)")


## Summary stats

In [24]:
print("=== Row counts ===")
print(df["category"].value_counts().to_string())

print("=== Row counts ===")
print(df["category"].value_counts().to_string())

print("\n=== Price stats (GBP) ===")
print(df.groupby("category")["price"].describe().round(2).to_string())

print("\n=== Condition distribution ===")
cond = df.groupby(["category", "condition"]).size().unstack(fill_value=0)
print(cond.to_string())

print("\n=== Buying options ===")
print(df["buying_options"].value_counts().head(10).to_string())

=== Row counts ===
category
smartwatches       500
laptops            500
speakers           500
vacuums            500
coffee             500
golf               500
cycling            500
pokemon_cards      500
vinyl_records      499
funko_pop          499
books              498
video_games        284
trainers           276
smartphones        236
gym_equipment      232
womens_clothing    138
textbooks           35
consoles            20
cameras             19
mens_clothing       17
lego                 4
handbags             1
=== Row counts ===
category
smartwatches       500
laptops            500
speakers           500
vacuums            500
coffee             500
golf               500
cycling            500
pokemon_cards      500
vinyl_records      499
funko_pop          499
books              498
video_games        284
trainers           276
smartphones        236
gym_equipment      232
womens_clothing    138
textbooks           35
consoles            20
cameras             19
m